# 🥁 Maracatu — Exploração

Notebook para explorar o corpus, o tokenizer, e testar o modelo treinado.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import torch
import sentencepiece as spm

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
print(f'MPS disponível: {torch.backends.mps.is_available()}')

## Explorar o corpus

In [ ]:
corpus_path = PROJECT_ROOT / 'data' / 'processed' / 'corpus.txt'

if corpus_path.exists():
    size_mb = corpus_path.stat().st_size / 1_000_000
    print(f'Corpus: {size_mb:.1f} MB')
    
    with open(corpus_path, encoding='utf-8') as f:
        preview = [next(f).strip() for _ in range(10)]
    
    for line in preview:
        if line:
            print(f'  {line[:120]}...' if len(line) > 120 else f'  {line}')
else:
    print('Corpus ainda não gerado. Rode: python scripts/clean_corpus.py')

## Explorar o tokenizer

In [ ]:
tokenizer_path = PROJECT_ROOT / 'tokenizer' / 'maracatu.model'

if tokenizer_path.exists():
    sp = spm.SentencePieceProcessor(model_file=str(tokenizer_path))
    print(f'Vocabulário: {sp.vocab_size():,} tokens')
    
    exemplos = [
        'O Brasil é um país tropical.',
        'Machado de Assis escreveu Dom Casmurro.',
        'Maracatu é um gênero pernambucano.',
    ]
    for texto in exemplos:
        tokens = sp.encode(texto, out_type=str)
        print(f'\n  "{texto}"')
        print(f'  → {tokens} ({len(tokens)} tokens)')
else:
    print('Tokenizer ainda não treinado. Rode: python tokenizer/train_tokenizer.py')

## Carregar modelo treinado

In [ ]:
from maracatu.model import Maracatu, ModelConfig
from maracatu.sample import load_model, get_device

checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'best.pt'

if checkpoint_path.exists():
    device = get_device()
    model, ckpt = load_model(checkpoint_path, device)
    print(f'Modelo carregado em {device}')
    print(f'Parâmetros: {model.num_params() / 1e6:.2f}M')
    print(f'Step: {ckpt["step"]:,}')
    print(f'Loss: {ckpt["loss"]:.4f}')
else:
    print('Checkpoint ainda não gerado. Rode o treino primeiro.')

## Gerar texto

In [ ]:
prompt = 'O Brasil é'
prompt_ids = torch.tensor([sp.encode(prompt)], dtype=torch.long, device=device)

output = model.generate(prompt_ids, max_new_tokens=100, temperature=0.8, top_k=50)
print(sp.decode(output[0].tolist()))